# Kaggriculture: Agent Selection Harness & Engine Analysis

Competition run, August-September 2026. Rating went from 300 to a peak of 2,666.

This notebook is not an agent. It's the measurement system I built to choose between
agents, plus the engine behavior that made the measurement trustworthy.

My local tests disagreed with the ladder four separate times before I found why. Each
one silently invalidated days of work. Those four traps are documented below, along
with a replay analysis showing what actually decides a game.

## Four traps that break local testing

**1. Last-callable entrypoint.** Kaggle's file runner picks the last callable bound in
the module namespace, not the function named `agent`. A notebook ending on an assignment
passes every turn and banks exactly the starting money. This cost me a submission that
scored 143. Rebinding the entrypoint took the same file to 1,705.

**2. Package version.** The competition ran engine 1.32.7. My notebook defaulted to
1.32.6, and every published agent went bankrupt locally while scoring normally on the
ladder.

**3. Config defaults.** Package defaults were `farmHandCostMult 10, startingMoney 2000`.
The competition used `1` and `3000`. Check with `starter` against `pass`: it must return
`[3509, 3000]`.

**4. Module state.** Trajectory agents hold globals across runs. Load a fresh module per
game or the second game is poisoned by the first.

In [1]:
"""Agent selection harness for Kaggriculture.

Kaggle loads a submission by file path and selects the last callable
bound in the module namespace. Always health-check by path before
submitting. An agent that returns exactly the starting money is
passing every turn, not playing badly.
"""
import glob
import statistics
from kaggle_environments import make

# Competition config. Package defaults differ and will silently
# produce wrong results. Verify with starter_vs_pass() below.
CFG = {
    "episodeSteps": 720,
    "farmHandCostMult": 1,
    "startingMoney": 3000,
    "townCenterSellInterval": 12,
    "townShopSellInterval": 4,
    "townShopUnlockInterval": 3,
}


def starter_vs_pass():
    """Config check. Must return [3509, 3000] on engine 1.32.7."""
    e = make("kaggriculture", configuration=dict(CFG, seed=0))
    e.run(["starter", "pass"])
    return [round(s.reward) for s in e.steps[-1]]


def health_check(path):
    """Return final bank against a passive opponent.

    A value at or near 3000 means the agent never acted, usually a
    last-callable entrypoint problem rather than a weak strategy.
    """
    e = make("kaggriculture", configuration=dict(CFG, seed=0))
    e.run([path, "pass"])
    return e.steps[-1][0].reward


def head_to_head(candidate, baseline, seeds=(0, 1, 2)):
    """Play candidate against baseline in both seats across seeds.

    Returns (wins, games, mean money margin). Margin below about
    5,000 has not moved ladder rating in practice.
    """
    wins = 0
    games = 0
    margins = []
    for seed in seeds:
        for seat in (0, 1):
            e = make("kaggriculture", configuration=dict(CFG, seed=seed))
            pair = [candidate, baseline] if seat == 0 else [baseline, candidate]
            e.run(pair)
            final = e.steps[-1]
            mine = final[seat].reward
            theirs = final[1 - seat].reward
            margins.append(mine - theirs)
            wins += mine > theirs
            games += 1
    return wins, games, statistics.mean(margins)


def rank_candidates(baseline, folder="/kaggle/input/notebooks"):
    """Test every agent in folder against baseline, best first."""
    results = []
    for path in sorted(set(glob.glob(folder + "/**/*.py", recursive=True))):
        if path == baseline:
            continue
        if health_check(path) <= 3100:
            continue
        wins, games, margin = head_to_head(path, baseline)
        if wins > games / 2:
            results.append((wins, margin, path))
            print(wins, "/", games, round(margin), path, flush=True)
    return sorted(results, reverse=True)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [2]:
print("config check:", starter_vs_pass(), "(must be [3509, 3000])")

config check: [3509, 3000] (must be [3509, 3000])


## What decides a game

Two of my own games, same agent, same tile layout, same sales volumes:

| | Best (183,898) | Worst (29,521) |
|---|---|---|
| Strawberry sold | 254 | 252 |
| Milk sold | 279 | 278 |
| Strawberry price | 236 | 3 |
| Milk price | 278 | 1 |

Identical production, 6x difference in bank. The town shop draw sets which products have
buyers, and price follows demand almost exactly. Premium demand of 3 gave 42k; demand of
13 gave 134k.

## Ranking by money picks the wrong agent

One agent scored highest against a passive opponent (191,525) and won 7 of 30 contested
games. Another scored lowest (86,837) and won 29 of 30. Rating comes from wins, so
contested win rate is the metric, not bank.

Margin size predicts rating movement:

| Local margin | Rating change |
|---|---|
| +31,577 | +500 |
| +3,346 | +150 |
| +1,921 | none |

Below roughly +5,000 is noise.

In [3]:
"""Replay analysis for Kaggriculture.

Score is set mostly by the town shop draw rather than by farming
quality. These functions measure that from downloaded replays.
"""
import collections
import json

SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}


def load(path):
    raw = json.load(open(path))
    return raw["steps"] if isinstance(raw, dict) and "steps" in raw else raw


def shop_demand(steps):
    """Total shop demand per product. Single-product shops count 2x."""
    last = steps[-1][0].get("observation") or {}
    shops = list((last.get("town") or {}).get("unlocked_shops") or [])
    demand = collections.Counter()
    for shop in shops:
        products = SHOP_PRODUCTS.get(shop, ())
        for product in products:
            demand[product] += 2 if len(products) == 1 else 1
    return demand


def final_prices(steps):
    last = steps[-1][0].get("observation") or {}
    return (last.get("market") or {}).get("prices") or {}


def sold_by_player(steps, player):
    """Units sold per product, read from the action stream."""
    sold = collections.Counter()
    for step in steps:
        action = (step[player] or {}).get("action") if player < len(step) else None
        if not isinstance(action, dict):
            continue
        for order in action.get("market") or []:
            if order and order[0] == "SELL" and len(order) > 2:
                sold[order[1]] += order[2]
    return sold

## Attribution

My final submissions used agents published by other competitors under Kaggle's code
sharing rules. The harness, the engine findings and the replay analysis in this notebook
are my own work.